In [1]:
from langchain_openai import ChatOpenAI

In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [4]:
from langchain.tools import tool

In [5]:
from typing import List, Union, Optional, Dict, Literal

In [158]:
import tiktoken

In [6]:
import os
from dotenv import load_dotenv

In [7]:
load_dotenv()

True

In [8]:
llm = ChatOpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("LLM")
)

In [40]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [41]:
tools = [calculator]
tool_map = {tool.name: tool for tool in tools}

In [42]:
import json

In [162]:
class Agent:
    def __init__(self, model: ChatOpenAI, tools=List):
        self.model = model
        self.tools = tools
        if self.tools:
            self.model = self.model.bind_tools(self.tools)
        self.tool_map = {tool.name: tool for tool in self.tools}
            
    def invoke(self, thread: Thread, self_append: bool = True):
        if thread.tail is not None:
            thread = thread.tail
        thread.agent = self
        while True:
            response = self.model.invoke(thread.messages)
            
            if not self_append:
                thread.agent = None
                return response
                
            thread.append(response)
            if response.tool_calls:
                for tool in response.tool_calls:
                    args=tool["args"]
                    call_id = tool["id"]
                    name = tool["name"]
                    result = self.tool_map[name].invoke(args)
                    thread.append(ToolMessage(name=name, content=result, tool_call_id=call_id))
            else:
                thread.agent = None
                return response

    def __ror__(self, thread: Thread):
        return self.invoke(thread)

In [163]:
class Thread:
    def __init__(
        self, 
        messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]] = None, 
        system_prompt: Union[str, SystemMessage] = None,
        compression_prompt: str = None,
        token_limit: int = None
    ):
        self.messages = []
        self.system_prompt = system_prompt
        self.compression_prompt = compression_prompt
        self.token_limit = token_limit
        self.agent = None
        self.encoder = tiktoken.encoding_for_model("gpt-4o-mini")
        self.root = None
        self.parent = None
        self.child = None
        self.tail = None

        if messages is not None:
            index = self._find_system_message(messages)
            if index == -1 or index == 0:
                self.messages = messages
            else:
                raise f"system message not at the starting, it was found at {index} index"
                    
        if self.system_prompt is not None:
            index = self._find_system_message(self.messages)
            if isinstance(self.system_prompt, str):
                self.system_prompt = SystemMessage(self.system_prompt)
            if index == 0:
                if len(self.messages) == 0:
                    self.append(self.system_prompt)
                else:
                    self[0] = self.system_prompt
            elif index == -1:
                self.messages = [self.system_prompt] + self.messages
            else:
                pass

    def _find_system_message(self, messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]]):
        for i in range(len(messages)):
            if isinstance(messages[i], SystemMessage):
                return i
        return -1

    def count_token(self):
        if self.tail is not None:
            content = [m.content for m in self.tail]
        else:
            content = [m.content for m in self]
        merged = "\n".join(content)
        return len(self.encoder.encode(merged))
        
    def append(self, message: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        token_usuage = self.count_token()
        if token_usuage > self.token_limit and self.agent is not None and self.compression_prompt is not None:
            self.messages.append(HumanMessage(self.compression_prompt))
            compression_report = self.agent.invoke(self, self_append=False).content
            self.messages.pop()
            
            new_thread = self.copy()
            self.child = new_thread
            new_thread.parent = self
            new_thread.root = self.root if self.root is not None else self
            self.root.tail = new_thread

            new_thread.messages = []
            if isinstance(self.root[0], SystemMessage):
                new_thread.append(self.root[0])
            new_thread.append(HumanMessage(compression_report)) 
        else:
            self.messages.append(message)

    def count(self):
        counts = {
            "depth":0,
            "system":0, 
            "human":0,
            "ai":0,
            "tool":0
        }
        thread = self
        depth = 0
        if isinstance(thread[0], SystemMessage):
            counts["system"]=1
        while True:
            for m in thread:
                if isinstance(m, AIMessage):
                    counts["ai"]+=1
                elif isinstance(m, HumanMessage):
                    counts["human"]+=1
                elif isinstance(m, ToolMessage):
                    counts["tool"]+=1
                else:
                    pass
            if thread.child is None:
                break
            else:
                thread = thread.child
                depth += 1
        counts["depth"] = depth
        return counts

    def __ror__(self, other: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.tail is not None:
            self.tail.append(other)
        else:
            self.append(other)

    # def __add__(self, other: Thread):
    #     new_thread = Thread()
    #     new_thread.messages = self.messages + other.messages
    #     return new_thread

    def __str__(self):
        counts = self.count()
        return json.dumps(counts)

    def __repr__(self):
        counts = self.count()
        return json.dumps(counts)

    def __iter__(self):
        if self.tail is not None:
            for msg in self.tail.messages:
                yield msg
        else:
            for msg in self.messages:
                yield msg
                
    def __getitem__(self, index):
        if self.tail is not None:
            return self.tail.messages[index]
        else:
            return self.messages[index]

    def __len__(self):
        if self.tail is not None:
            return len(self.tail.messages)
        else:
            return len(self.messages)

    def __setitem__(self, index, value):
        if self.tail is not None:
            self.tail.messages[index] = value
        else:
            self.messages[index] = value

    def __copy__(self):
        new_instance = Thread()
        return new_instance
        

In [148]:
myagent = Agent(
    model=llm,
    tools=tools
)

In [149]:
thread = Thread()

In [150]:
thread.append(SystemMessage("You are a Mathematical Expression solver"))

In [151]:
HumanMessage("Solve this particular expression : 3*4+77 use calculator tool") | thread

In [152]:
thread.messages

[SystemMessage(content='You are a Mathematical Expression solver', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Solve this particular expression : 3*4+77 use calculator tool', additional_kwargs={}, response_metadata={})]

In [153]:
response = thread | myagent

In [154]:
response.content

'The result of the expression \\(3 \\times 4 + 77\\) is **89**.'

In [155]:
print(thread)

{"system": 1, "human": 1, "ai": 2, "tool": 1}


In [156]:
thread

{"system": 1, "human": 1, "ai": 2, "tool": 1}

In [157]:
thread.messages

[SystemMessage(content='You are a Mathematical Expression solver', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Solve this particular expression : 3*4+77 use calculator tool', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 297, 'total_tokens': 408, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-8aef49d502364437', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03312-3d54-7961-8096-6c52d52671c8-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '3*4+77'}, 'id': 'chatcmpl-tool-a486495d02d82ae5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 297, 'output_tokens': 111, 'total_tokens': 408, 'input_token_details': {}, 'output_token

In [39]:
for m in response:
    if isinstance(m, SystemMessage):
        print("system message:\n")
    elif isinstance(m, HumanMessage):
        print("human message:\n")
    elif isinstance(m, AIMessage):
        print("ai message:\n")
    elif isinstance(m, ToolMessage):
        print("tool message:\n")
    else:
        print("other type:\n")
    print(m.model_dump_json())
    print("\n\n")

system message:

{"content":"You are a Mathematical Expression solver","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}



human message:

{"content":"Solve this particular expression : 3*4+77 use calculator tool","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null}



ai message:

{"content":"","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":70,"prompt_tokens":297,"total_tokens":367,"completion_tokens_details":null,"prompt_tokens_details":null},"model_provider":"openai","model_name":"Deepseek-vapt","system_fingerprint":"vllm-0.25.0-tp2-ep-d4f8ac0c","id":"chatcmpl-bc18df2e93c1cfe1","finish_reason":"tool_calls","logprobs":null},"type":"ai","name":null,"id":"lc_run--01a03256-a5d0-7433-8fa3-9299a904fa75-0","tool_calls":[{"name":"calculator","args":{"expression":"3*4+77"},"id":"chatcmpl-tool-8e2f16389057886f","type":"tool_call"}],"invalid_tool_calls":[],"usage_metadata":{"input_t

In [19]:
s = SystemMessage("hii")

In [20]:
s.model_dump_json()

'{"content":"hii","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}'

In [45]:
response["messages"][-1].content

'The result of the expression \\(3 \\times 4 + 77\\) is **89**.'

In [47]:
response["messages"].append(HumanMessage("Now also calculate this one 456%100"))

In [54]:
nr = agent.invoke(nr)